In [ ]:
# Minimal setup: load env, create client
import os
from dotenv import load_dotenv
from md_python import MDClient, Upload, ExperimentDesign, SampleMetadata

def create_md_client_from_env() -> MDClient:
    """Create MD v2 client using MD_AUTH_TOKEN (and optionally MD_API_BASE_URL) from .env"""
    load_dotenv()
    api_token = os.getenv("MD_AUTH_TOKEN")
    assert api_token, "MD_AUTH_TOKEN not set (check your .env)"
    base_url = os.getenv("MD_API_BASE_URL")  # optional — defaults to production
    return MDClient(api_token=api_token, base_url=base_url)


client = create_md_client_from_env()
print("Client initialized.")

In [ ]:
# Print current md-python tag
import importlib.metadata
import json
import subprocess
from pathlib import Path

direct_url_files = [p for p in importlib.metadata.files("md-python") if "direct_url.json" in str(p)]
if direct_url_files:
    direct_url = json.loads(direct_url_files[0].read_text())
    url = direct_url.get("url", "")
    if "tags/" in url:
        tag = url.rstrip(".tar.gz").split("/")[-1]
    elif url.startswith("file://"):
        repo_path = url[len("file://"):]
        result = subprocess.run(
            ["git", "describe", "--tags"],
            cwd=repo_path, capture_output=True, text=True
        )
        tag = result.stdout.strip() or "(no tag)"
    else:
        tag = "(unknown)"
    print(tag)
else:
    print("(no install metadata found)")

In [ ]:
# Quick connectivity test
health = client.health.check()
print("Health:", health)
assert str(health.get("status", "")).lower() in {"healthy", "ok", "pass"}, "API health check failed"

In [ ]:
from pathlib import Path
from typing import Iterable, List, Optional

def discover_local_files(directory: Path, allowed_exts: Optional[set[str]] = None, override: Optional[Iterable[str]] = None) -> List[str]:
    """Return filenames to upload, using override or discovering by extension."""
    if override:
        return list(override)
    exts = allowed_exts or {".tsv", ".txt", ".csv", ".d", ".log"}
    files = [p.name for p in sorted(directory.iterdir()) if p.is_file() and p.suffix.lower() in exts]
    assert files, f"No matching files found in: {directory}"
    return files

# Use test_data under development/GI/do_not_save
LOCAL_DATA_DIR = Path("/Users/giuseppeinfusini/wd/Data_for_upload_md/MD-format/Test_data/A549/subdataset_01").resolve()
assert LOCAL_DATA_DIR.exists(), f"Path not found: {LOCAL_DATA_DIR}"

# Upload only the proteomics TSV files (design/metadata CSVs are read locally, not uploaded)
OVERRIDE_FILENAMES = [
    "proteomics_proteins_COMBINED.tsv",
    "proteomics_peptides_COMBINED.tsv",
]

filenames = discover_local_files(LOCAL_DATA_DIR, override=OVERRIDE_FILENAMES)
print("Files to upload:", filenames)

In [ ]:
import csv
from pathlib import Path
from typing import List, Tuple

DESIGN_CSV_NAME = "experiment_design_COMBINED.csv"


def _header_lower(header: List[str]) -> List[str]:
    return [h.strip().lower() if isinstance(h, str) else "" for h in header]


def _find_column_index(header_lower: List[str], names: Tuple[str, ...]) -> int:
    for n in names:
        if n in header_lower:
            return header_lower.index(n)
    raise ValueError(f"CSV must have one of {names}; got {header_lower}")


def load_design_and_metadata_from_csv(
    csv_path: Path, _filenames: List[str]
) -> Tuple[ExperimentDesign, SampleMetadata]:
    """Read experiment_design.csv and build ExperimentDesign and SampleMetadata from it.
    Uses all data rows from the CSV. Sample metadata is one row per unique sample_name.
    """
    with open(csv_path, "r", encoding="utf-8") as f:
        raw = list(csv.reader(f))
    if not raw:
        raise ValueError(f"{csv_path.name} is empty")
    header = [h.strip() for h in raw[0]]
    hl = _header_lower(header)
    idx_fn = _find_column_index(hl, ("filename", "file"))
    idx_sample = _find_column_index(hl, ("sample_name", "sample"))
    idx_cond = _find_column_index(hl, ("condition", "group"))

    data_rows = [r for r in raw[1:] if isinstance(r, list)]
    design_data = [["filename", "sample_name", "condition"]]
    for r in data_rows:
        design_data.append([
            r[idx_fn].strip() if len(r) > idx_fn else "",
            r[idx_sample].strip() if len(r) > idx_sample else "",
            r[idx_cond].strip() if len(r) > idx_cond else "",
        ])
    exp_design = ExperimentDesign(data=design_data)

    sample_col_indices = [
        i for i, h in enumerate(header)
        if hl[i] not in ("filename", "file")
    ]
    sample_headers = [header[i] for i in sample_col_indices]
    seen = set()
    sample_rows = []
    for r in data_rows:
        sn = r[idx_sample].strip() if len(r) > idx_sample else ""
        if sn and sn not in seen:
            seen.add(sn)
            sample_rows.append([
                r[i].strip() if len(r) > i else "" for i in sample_col_indices
            ])
    sample_md = SampleMetadata(data=[sample_headers] + sample_rows)
    return exp_design, sample_md


design_csv_path = LOCAL_DATA_DIR / DESIGN_CSV_NAME
assert design_csv_path.exists(), f"Missing {DESIGN_CSV_NAME} in {LOCAL_DATA_DIR}"
exp_design, sample_md = load_design_and_metadata_from_csv(design_csv_path, filenames)

assert exp_design.data and exp_design.data[0] == ["filename", "sample_name", "condition"]
assert sample_md.data
print("Design and metadata from experiment_design.csv:")
print("  design rows:", len(exp_design.data) - 1)
print("  sample metadata rows:", len(sample_md.data) - 1)

In [ ]:
# sample_md stays as loaded (strings only). Dose is converted to numbers in DoseResponseDataset when creating the dose response job.

In [ ]:
exp_design

In [ ]:
sample_md

In [ ]:
from datetime import datetime

upload_name = f"Local Upload {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
upload_name = "Upload_v2_test_0.2.2"
upload_source = os.getenv("MD_DEFAULT_SOURCE", "md_format")
labelling_method = os.getenv("MD_LABELLING_METHOD", "lfq")

upload = Upload(
    name=upload_name,
    source=upload_source,
    labelling_method=labelling_method,
    experiment_design=exp_design,
    sample_metadata=sample_md,
    file_location=str(LOCAL_DATA_DIR),
    filenames=filenames,
)

upload_id = client.uploads.create(upload)
print("Upload created. ID:", upload_id)

In [ ]:
import time

# Optional: poll until the upload reaches a terminal state
# upload_id = "..."  # uncomment and set if resuming
_t0 = time.monotonic()
completed_upload = client.uploads.wait_until_complete(upload_id, poll_s=10, timeout_s=3600)
_elapsed = time.monotonic() - _t0
print(f"Upload completed in {_elapsed:.1f}s ({_elapsed/60:.1f} min)")
print(completed_upload)

## Create dose response and pairwise datasets

After the experiment has completed, get the initial intensity dataset and use it to create:
1. **Dose response analysis** dataset (via API job)
2. **Pairwise comparison** dataset (via `PairwiseComparisonDataset`)

### Get initial intensity dataset

Required as input for both dose response and pairwise jobs. Uses the experiment's intensity dataset (same name as experiment).

In [ ]:
dataset = client.datasets.find_initial_dataset(upload_id)
print("Initial dataset ID:", dataset.id)
dataset.id

Initial dataset ID: f033d49a-c65a-4699-9b5b-914952eb1761
UUID('f033d49a-c65a-4699-9b5b-914952eb1761')

### Dose response analysis

Create a dose response dataset using `DoseResponseDataset`. Sample names come from sample metadata; control samples are typically zero-dose (e.g. dose=0). Adjust `control_samples` to match your design.

**Get JSON payload for manual request**

Run the cell below to print the request body that `client.datasets.create()` sends. Copy the output to try with curl or Postman (use your `MD_AUTH_TOKEN` and `MD_API_BASE_URL`).

**experiment_design** (for manual `job_run_params`)

Add this inside `job_run_params` when calling the API manually. Column names → list of values; **dose must be numbers**. Example for first 8 samples (MD00001):

```json
"experiment_design": {
  "sample_name": ["MD00001_0", "MD00001_0.01", "MD00001_0.1", "MD00001_1", "MD00001_10", "MD00001_100", "MD00001_1000", "MD00001_10000"],
  "condition": ["md00001_a", "md00001_a", "md00001_b", "md00001_b", "md00001_b", "md00001_a", "md00001_b", "md00001_a"],
  "dose": [0, 0.01, 0.1, 1, 10, 100, 1000, 10000],
  "cellline": ["HEK293", "HEK293", "HEK293", "HEK293", "HEK293", "HEK293", "HEK293", "HEK293"],
  "family": ["FAMILY_A", "FAMILY_A", "FAMILY_A", "FAMILY_A", "FAMILY_A", "FAMILY_A", "FAMILY_A", "FAMILY_A"],
  "drug": ["MD00001", "MD00001", "MD00001", "MD00001", "MD00001", "MD00001", "MD00001", "MD00001"]
}
```

Full `job_run_params` with experiment_design: include `control_samples`, `log_intensities`, `use_imputed_intensities`, `normalise`, `span_rollmean_k`, `prop_required_in_protein`, and `experiment_design` as above.

In [ ]:
from md_python import DoseResponseDataset

drc_dataset_name = "Dose response test_data"

# Sample names from sample metadata (first column is sample_name)
sample_names = sample_md.to_columns().get("sample_name", [])
# Control samples: e.g. zero-dose for test_data (dose=0)
dose_col = sample_md.to_columns().get("dose", [])
control_samples = [
    sn for sn, d in zip(sample_names, dose_col) if str(d).strip() == "0"
]
if not control_samples:
    control_samples = sample_names[:2]

drc_builder = DoseResponseDataset(
    input_dataset_ids=[str(dataset.id)],
    dataset_name=drc_dataset_name,
    sample_names=sample_names,
    control_samples=control_samples,
    sample_metadata=sample_md,
    dose_column="dose",
    log_intensities=True,
    use_imputed_intensities=True,
    normalise="none",
    span_rollmean_k=1,
    prop_required_in_protein=0.5,
)
drc_dataset_id = drc_builder.run(client)
print("Dose response dataset ID:", drc_dataset_id)
drc_dataset_id

In [ ]:
# Optional: get the exact JSON payload to try the request manually (e.g. curl/Postman)
# v2 uses a flat payload (no "dataset" wrapper key)
import json

ds = drc_builder.to_dataset()
payload = {
    "input_dataset_ids": [str(x) for x in ds.input_dataset_ids],
    "name": ds.name,
    "job_slug": ds.job_slug,
    "job_run_params": ds.job_run_params or {},
}
if ds.sample_names is not None:
    payload["sample_names"] = ds.sample_names

print(json.dumps(payload, indent=2))

In [ ]:
drc_state = client.datasets.wait_until_complete(
    upload_id=str(upload_id),
    dataset_id=str(drc_dataset_id),
    poll_s=10,
    timeout_s=3600,
)
print("Dose response state:", drc_state.state)
drc_state

### Pairwise analysis

Define comparisons (e.g. all conditions vs one control) and create a pairwise comparison dataset using `PairwiseComparisonDataset`.

In [ ]:
from md_python import PairwiseComparisonDataset

condition_column = "condition"
control_condition = "md00001_a"
pairwise_dataset_name = "Pairwise test_data"

comparisons = PairwiseComparisonDataset.pairwise_vs_control(sample_md, column=condition_column, control=control_condition)
print("Comparisons (case vs control):", comparisons)

pw = PairwiseComparisonDataset(
    input_dataset_ids=[str(dataset.id)],
    dataset_name=pairwise_dataset_name,
    sample_metadata=sample_md,
    condition_column=condition_column,
    condition_comparisons=comparisons,
    entity_type="protein",
)
pairwise_dataset_id = pw.run(client)
print("Pairwise dataset ID:", pairwise_dataset_id)
pairwise_dataset_id

In [ ]:
comparisons

In [ ]:
pw_state = client.datasets.wait_until_complete(
    upload_id=str(upload_id),
    dataset_id=str(pairwise_dataset_id),
    poll_s=10,
    timeout_s=3600,
)
print("Pairwise state:", pw_state.state)
pw_state

In [ ]:
completed_upload